# Plotting

vorflow's three pipeline classes each have a `plot()` method:

- `ConceptualMesh.plot()` draws the conceptual model: polygons coloured by
  `zone_id`, with lines and points on top;
- `MeshGenerator.plot()` draws the triangular (or mixed tri/quad) mesh;
- `VoronoiTessellator.plot()` draws the Voronoi cells.

Each takes an optional `ax` and returns the axes, so plots can be layered and
styled with Matplotlib. Matplotlib is optional; install it with
`pip install vorflow[plot]`.

In [ ]:
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point, box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, set_verbosity

set_verbosity(0)

## 1. Conceptual model with legend labels

Every `add_*` method takes an optional `label`. `plot()` gives each label one
legend entry, so `ax.legend()` needs no arguments. Features sharing a label
share an entry, and unlabelled features get none.

Embedded polygons are filled by `zone_id`. Lines are drawn in the `color`
given to `add_line`; without one, barriers are red and other lines blue.
Field-only (`embed=False`) features, which only size the mesh, are drawn
as dashed outlines, dashed lines and hollow points.

In [ ]:
domain = box(0, 0, 200, 200)
zone = box(120, 120, 190, 190)
refinement = box(20, 120, 70, 170)

cm = ConceptualMesh()
cm.add_polygon(domain, zone_id=1, label="Aquifer")
cm.add_polygon(zone, zone_id=2, resolution=8, z_order=1, label="Clay lens")
cm.add_polygon(refinement, zone_id=3, resolution=6, embed=False, label="Refinement (6 m)")
cm.add_line(LineString([(100, 0), (100, 150)]), line_id="Fault-1", resolution=2,
            is_barrier=True, label="Fault")
cm.add_line(LineString([(0, 60), (200, 40)]), line_id="River-1", resolution=4, label="River",
            color="tab:blue")
cm.add_line(LineString([(0, 100), (60, 100)]), line_id="Drain-1", resolution=4, label="Drains",
            color="tab:green")
cm.add_line(LineString([(140, 0), (200, 60)]), line_id="Drain-2", resolution=4, label="Drains",
            color="tab:green")
cm.add_point(Point(25, 25), point_id="Well-A", resolution=2, label="Wells")
cm.add_point(Point(160, 100), point_id="Well-B", resolution=2, label="Wells")

fig, ax = plt.subplots(figsize=(7, 7))
cm.plot(ax=ax)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
ax.set_title("Registered features");

Polygon keywords such as `alpha` or `edgecolor` are passed to
`GeoDataFrame.plot`, and `column=None` draws all polygons in one colour.

`clean=True` plots the features as `generate()` left them instead: overlaps
resolved (the lens is cut out of the aquifer), lines snapped and clipped to
the domain. Labels survive cleaning.

In [ ]:
cm.generate()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
cm.plot(ax=axes[0], alpha=0.3)
axes[0].set_title("Raw, alpha=0.3")
cm.plot(ax=axes[1], clean=True, column=None, color="wheat")
axes[1].set_title("Clean, one colour")
axes[1].legend(loc="lower right", fontsize="small");

## 2. Triangular mesh

`MeshGenerator.plot()` draws the element grid from the last `generate()`.
With `column=None` (the default) only the outlines are drawn; pass a column
of the element grid, such as `zone_id`, to fill the elements.

In [ ]:
mesher = MeshGenerator(background_lc=15, verbosity=0, conceptual_mesh=cm)
mesher.generate()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
mesher.plot(ax=axes[0])
axes[0].set_title("Elements")
mesher.plot(ax=axes[1], column="zone_id", cmap="Pastel1", categorical=True)
axes[1].set_title("Elements by zone");

## 3. Voronoi grid

`VoronoiTessellator.plot()` colours the cells by `zone_id` by default;
`show_generators=True` also draws each cell's generator point. Since every
`plot()` returns its axes, the conceptual model can be drawn over the grid
to check how the cells follow the features.

In [ ]:
tessellator = VoronoiTessellator(mesher)
grid = tessellator.generate()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
tessellator.plot(ax=axes[0], cmap="Pastel1", categorical=True)
axes[0].set_title(f"{len(grid)} Voronoi cells")

tessellator.plot(ax=axes[1], column=None, show_generators=True)
# Hollow, borderless polygons: only the lines and points show over the cells.
cm.plot(ax=axes[1], clean=True, facecolor="none", edgecolor="none")
axes[1].set_xlim(70, 130)
axes[1].set_ylim(20, 80)
axes[1].set_title("Cells and generators at the fault");

Near the fault the generators sit in pairs either side of the line, so the
cell faces between them follow it: the barrier is a set of cell faces, not a
row of cells.

The grid is a GeoDataFrame, so any column can be plotted the same way, for
example the cell area.

In [ ]:
grid["area"] = grid.area

fig, ax = plt.subplots(figsize=(7, 7))
tessellator.plot(ax=ax, column="area", cmap="viridis", legend=True,
                 legend_kwds={"label": "Cell area (m²)", "shrink": 0.7})
ax.set_title("Cell area");